# ML-10 — Content Action Playbook

This notebook translates our validated model probabilities and Week-4 rule signals into an operational **Content Action Playbook** with ranked queues, reason codes, confidence levels, and human review guardrails.

## 1. Ranked actions + reason codes

*Below are both the preserved Week-4 Top-10 queue ($N = 30$) and the held-out client Top-10 queue ($n_{\text{test}} = 2,325$).* 

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

def find_repo_root() -> Path:
    cur = Path.cwd().resolve()
    for p in [cur, *cur.parents]:
        if (p / "data" / "raw" / "content_refresh_anonymized.csv").exists():
            return p
    return cur

REPO_ROOT = find_repo_root()
import json

with open(REPO_ROOT / "work" / "outputs" / "capstone_metrics.json", "r", encoding="utf-8") as f:
    M = json.load(f)

print("=== Table A: Preserved Week-4 Baseline Top-10 Queue (N=30) ===")
w04_q = pd.DataFrame(M["w04_top10_queue"])
print(w04_q[["rank", "item_id", "impressions", "clicks", "position", "staleness_days", "score", "action", "reason_code", "confidence"]].to_string(index=False))

print("\n=== Table B: FlyRank 30k Unseen Client Holdout Top-10 Queue (n_test=2,325) ===")
hold_q = pd.DataFrame(M["flyrank_30k_evaluation"]["top10_queue_client_holdout"])
print(hold_q[["rank", "content_id", "priority_score", "pred_prob", "score", "action_full", "is_declining_label"]].to_string(index=False))


=== Table A: Preserved Week-4 Baseline Top-10 Queue (N=30) ===
 rank  item_id  impressions  clicks  position  staleness_days  score     action       reason_code                             confidence
    1 item_002         3892     117        11              25      5 REVIEW_NOW STALE_HIGH_VOLUME   High (all 3 baseline thresholds met)
    2 item_011         2679      86        12              25      5 REVIEW_NOW STALE_HIGH_VOLUME   High (all 3 baseline thresholds met)
    3 item_015         3615      38        10              24      5 REVIEW_NOW STALE_HIGH_VOLUME   High (all 3 baseline thresholds met)
    4 item_019         4214     340         8              15      5 REVIEW_NOW STALE_HIGH_VOLUME   High (all 3 baseline thresholds met)
    5 item_020         2306     390        10              22      5 REVIEW_NOW STALE_HIGH_VOLUME   High (all 3 baseline thresholds met)
    6 item_024         4641     236        12              14      5 REVIEW_NOW STALE_HIGH_VOLUME   High (all 3 bas

## 2. Intended use and limits

- **Intended Use:** Weekly editorial triage—helping SEO analysts and managing editors decide which 10 to 50 pages to inspect first.
- **Limits:** Predictions measure historical association with content decline (`is_declining_label`), not causal proof that editing a page will increase traffic.

In [2]:
playbook_table = pd.DataFrame([
    {"Reason_Code": "STALE_HIGH_VOLUME", "Rule_Trigger": "impressions >= 1000 & staleness_days >= 14", "Recommended_Human_Check": "Check query trend & update outdated sections/examples"},
    {"Reason_Code": "STRIKING_DISTANCE_DECAY_RISK", "Rule_Trigger": "position >= 8 & impressions >= 500", "Recommended_Human_Check": "Audit subtopic coverage & internal links to push toward Page 1"},
    {"Reason_Code": "LOW_CTR_HIGH_EXPOSURE", "Rule_Trigger": "impressions >= 500 & ctr_pct < 0.50%", "Recommended_Human_Check": "Inspect SERP layout (AI Overviews) & test title/meta snippet"},
    {"Reason_Code": "HIGH_MODEL_DECLINE_RISK", "Rule_Trigger": "pred_prob >= 0.65", "Recommended_Human_Check": "Priority editorial review combining log-CTR gap and staleness"},
])
print(playbook_table.to_string(index=False))


                 Reason_Code                               Rule_Trigger                                        Recommended_Human_Check
           STALE_HIGH_VOLUME impressions >= 1000 & staleness_days >= 14          Check query trend & update outdated sections/examples
STRIKING_DISTANCE_DECAY_RISK         position >= 8 & impressions >= 500 Audit subtopic coverage & internal links to push toward Page 1
       LOW_CTR_HIGH_EXPOSURE       impressions >= 500 & ctr_pct < 0.50%   Inspect SERP layout (AI Overviews) & test title/meta snippet
     HIGH_MODEL_DECLINE_RISK                          pred_prob >= 0.65  Priority editorial review combining log-CTR gap and staleness


## 3. Human review + the no-go list

1. **Sibling URL Cannibalization Check:** Before editing a declining page, check if another URL on the same domain absorbed its impressions.
2. **High-CTR Guardrail:** Pages on Page 1 with strong CTR (such as `item_005` with `19.99%` CTR or `item_063` with `749` clicks) must not be rewritten blindly just because `staleness_days >= 14`.
3. **No Unattended Automation:** Never pipe model scores directly into automated LLM rewrites or URL deletion.

In [3]:
for item in hold_q.head(5).to_dict(orient="records"):
    print(f"Rank {item['rank']} ({item['content_id']}): score={item['priority_score']} | Caveat: {item['what_could_make_it_wrong']}")


Rank 1 (content_3e79eaafc89d): score=94.1 | Caveat: Traffic change could reflect seasonal demand shifts or sibling URL cannibalization rather than content staleness.
Rank 2 (content_8fdbff16a886): score=93.4 | Caveat: Traffic change could reflect seasonal demand shifts or sibling URL cannibalization rather than content staleness.
Rank 3 (content_477f7892c1f1): score=93.0 | Caveat: Page ranks on Page 1 with near-zero CTR; SERP features/AI overview or snippet mismatch may explain low clicks without content decay.
Rank 4 (content_ef731e95e774): score=93.0 | Caveat: Traffic change could reflect seasonal demand shifts or sibling URL cannibalization rather than content staleness.
Rank 5 (content_03582b12af32): score=92.6 | Caveat: Page ranks on Page 1 with near-zero CTR; SERP features/AI overview or snippet mismatch may explain low clicks without content decay.


## 4. Monitoring / retrain triggers

- **Base-Rate Shift Trigger:** Recalibrate thresholds if portfolio declining base rate shifts by more than $\pm 10$ percentage points (e.g., training clients `55.48%` vs. held-out clients `39.10%`).
- **Queue Precision Trigger:** Retrain if `Precision@20` on human-reviewed batches drops below `0.50` (validated benchmark: `0.7000`).

In [4]:
triggers = pd.DataFrame([
    {"Monitor_Signal": "Client Declining Base Rate", "Validated_Reference": "0.5548 train / 0.3910 test", "Trigger_Threshold": "+/- 0.10 shift", "Action": "Recalibrate probability cutoff per client"},
    {"Monitor_Signal": "Top-20 Queue Precision@20", "Validated_Reference": "0.7000 on unseen clients", "Trigger_Threshold": "< 0.5000", "Action": "Re-estimate 7-feature Logistic Regression weights"},
])
print(triggers.to_string(index=False))


            Monitor_Signal        Validated_Reference Trigger_Threshold                                            Action
Client Declining Base Rate 0.5548 train / 0.3910 test    +/- 0.10 shift         Recalibrate probability cutoff per client
 Top-20 Queue Precision@20   0.7000 on unseen clients          < 0.5000 Re-estimate 7-feature Logistic Regression weights


## 5. Exports for the paper

*Verify all exported JSON metrics and SVG charts used in `work/capstone_report.md` and `docs/index.html`.*

In [5]:
for svg_name in [
    "fig1_feature_distributions.svg",
    "fig2_baseline_vs_model_metrics.svg",
    "fig3_logistic_regression_coefficients.svg",
    "fig4_error_analysis_breakdown.svg",
    "fig5_top10_refresh_queue.svg",
]:
    p_work = REPO_ROOT / "work" / "figures" / svg_name
    p_docs = REPO_ROOT / "docs" / "figures" / svg_name
    assert p_work.exists() and p_docs.exists(), f"Missing {svg_name}"
    print(f"[VERIFIED] {svg_name} ({p_work.stat().st_size:,} bytes)")


[VERIFIED] fig1_feature_distributions.svg (188,074 bytes)
[VERIFIED] fig2_baseline_vs_model_metrics.svg (116,620 bytes)
[VERIFIED] fig3_logistic_regression_coefficients.svg (104,143 bytes)
[VERIFIED] fig4_error_analysis_breakdown.svg (113,304 bytes)
[VERIFIED] fig5_top10_refresh_queue.svg (152,970 bytes)


## Self-check

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/`